# Payments Data Transformation Pipeline

This notebook loads the **Transactions Fraud Datasets** from Kaggle into Databricks and transforms the raw data into analysis-ready tables. The new columns describe the **channel** of each transaction, the **spending level** of merchant cities, the **segment** of each customer and the **financial and risk profile** of each user.

**Dataset:** [Transactions Fraud Datasets](https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets) on Kaggle

## Pipeline Steps:

### 1. Data Loading (Python)
- Downloads the dataset with `kagglehub`
- Loads transactions for **2010 and 2011**, cards, users and fraud labels
- Cleans the transaction `amount` (text `$-77.00` → number `-77.0`)
- Saves everything as Delta tables in `payments.default`

### 2. Transaction Features (SQL) — `pay_transactions`
- **Channel:** `is_online` (ONLINE / OFFLINE)
- **City spending level:** `city_amount_level` (Low / Middle / High Level)
- **Customer segmentation:** `customer_frequency_level`, `customer_monetary_level`, `customer_boston_class`

### 3. User Features (SQL) — `pay_users`
- **Cleaning:** `yearly_income`, `total_debt`, `per_capita_income` (text → number)
- **Financial ratios:** `debt_to_income`, `income_vs_area`, `debt_per_card`, `years_to_retirement`
- **Bands:** `credit_band`, `generation`
- **Risk:** `risk_segment`, `fraud_risk_level`

### 4. Final Check
- Overview of the final tables and of the new columns

| Table | Grain (one row = ) | Columns added |
|-------|--------------------|---------------|
| `pay_transactions` | one transaction | 5 |
| `pay_users` | one user | 9 (+ 3 cleaned) |
| `pay_cards` | one card | – |
| `pay_labels` | one labelled transaction | – |

> **Run the notebook once, from top to bottom (Run all).** Every transformation rebuilds a table from itself, so running a single cell twice fails or duplicates data. To start again, rerun the whole notebook from step 1.

---
## 1. Data Loading

Install `kagglehub` and restart Python so the package is available.

In [0]:
%pip install --upgrade kagglehub

In [0]:
dbutils.library.restartPython()

### Download the dataset and read the files

The dataset has four files:
- `transactions_data.csv`: one row per card transaction (over 13 million rows, 2010–2019)
- `cards_data.csv`: one row per card
- `users_data.csv`: one row per customer
- `train_fraud_labels.json`: fraud label (`Yes` / `No`) for each transaction id

In [0]:
import kagglehub
import pandas as pd
import json

path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")
print("Path to dataset files:", path)

transactions = pd.read_csv(f"{path}/transactions_data.csv")
cards = pd.read_csv(f"{path}/cards_data.csv")
users = pd.read_csv(f"{path}/users_data.csv")

with open(f"{path}/train_fraud_labels.json", "r") as f:
    raw = json.load(f)

print("Transactions:", transactions.shape)
print("Cards:       ", cards.shape)
print("Users:       ", users.shape)

### Fraud labels: from nested JSON to a table

The JSON file has the form `{"target": {"7476734": "No", ...}}`. It is flattened into a table with two columns: `transaction_id` (converted to integer, to match the transaction `id`) and `is_fraud`.

In [0]:
labels = pd.DataFrame(
    list(raw["target"].items()),
    columns=["transaction_id", "is_fraud"]
)
labels["transaction_id"] = labels["transaction_id"].astype(int)
labels.head()

### Transactions 2010: filter and clean the amount

Only transactions from **2010** are kept at first, to keep the table small enough to work with.

The `amount` column is text with a dollar sign (`$-77.00`). `.str[1:]` removes the first character (`$`) and `.astype(float)` turns the rest into a number. Negative amounts are **refunds**.

In [0]:
transactions_2010 = transactions[transactions['date'].str.startswith('2010')].copy()
transactions_2010['amount'] = transactions_2010['amount'].str[1:].astype(float)
print("Transactions 2010:", transactions_2010.shape)
transactions_2010.head()

### Save the tables to Databricks (Delta)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS payments.default")

def save(pdf, table):
    spark.createDataFrame(pdf).write.mode('overwrite') \
        .option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: {spark.table(table).count():,} rows")

save(transactions_2010, "payments.default.pay_transactions")
save(cards, "payments.default.pay_cards")
save(users, "payments.default.pay_users")
save(labels, "payments.default.pay_labels")

### Add transactions 2011

The 2011 transactions get the same cleaning and are **appended** (`mode('append')`) to the table, which then contains 2010 and 2011.

> Run this cell only once: running it again adds 2011 a second time.

In [0]:
transactions_2011 = transactions[transactions['date'].str.startswith('2011')].copy()
transactions_2011['amount'] = transactions_2011['amount'].str[1:].astype(float)
transactions_2011_spark = spark.createDataFrame(transactions_2011)
transactions_2011_spark.write.mode('append').saveAsTable("payments.default.pay_transactions")

In [0]:
%sql
SELECT substr(date, 1, 4) AS year, COUNT(*) AS transactions, ROUND(SUM(amount), 0) AS total_amount
FROM payments.default.pay_transactions
GROUP BY substr(date, 1, 4)
ORDER BY year

---
## 2. Transaction Features — `pay_transactions`

### 2.1 Channel: `is_online`

**What:** marks each transaction as `ONLINE` or `OFFLINE`.

**How:** in this dataset, online purchases have no physical shop, so their `merchant_city` is the word `ONLINE`. Every other value is a real city, so the purchase happened in a shop.

**Why:** online and in-store payments behave differently (amounts, fraud patterns), so the channel is one of the most useful ways to split the data.

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS
SELECT *, CASE WHEN merchant_city = 'ONLINE' THEN 'ONLINE'
ELSE 'OFFLINE'
END AS is_online
FROM payments.default.pay_transactions;

**Check:** number of transactions and amount per channel.

In [0]:
%sql
SELECT is_online,
       COUNT(*)                                            AS transactions,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)  AS pct_transactions,
       ROUND(AVG(amount), 2)                               AS avg_amount
FROM payments.default.pay_transactions
GROUP BY is_online

### 2.2 City spending level: `city_amount_level`

**What:** classifies each merchant city as **Low**, **Middle** or **High Level** by the total money spent there.

**How**, in three steps:
1. **`city_totals`:** adds up `amount` for each `merchant_city`, skipping empty cities and missing amounts.
2. **`city_groups`:** sorts the cities from lowest to highest total and splits them into **three groups of equal size** with `NTILE(3)`: the bottom third becomes *Low Level*, the middle third *Middle Level* and the top third *High Level*.
3. **Final `SELECT`:** joins the level back to every transaction of that city with a `LEFT JOIN`. Transactions whose city has no level (empty or missing city) get `'Città non disponibile'` through `COALESCE`.

**Why:** it shows whether a transaction happens in a place where a lot of money is spent or in a minor location.

**Things to know:**
- The thirds are made over **cities**, not over transactions. Each group has the same number of cities, not the same number of transactions.
- `ONLINE` is treated like a city, and with its large total it is likely to end up in *High Level*.
- Refunds (negative amounts) reduce the city total.

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS

WITH city_totals AS (
    SELECT
        merchant_city,
        SUM(amount) AS city_total_amount
    FROM payments.default.pay_transactions
    WHERE merchant_city IS NOT NULL
      AND trim(merchant_city) <> ''
      AND amount IS NOT NULL
    GROUP BY merchant_city
),

city_groups AS (
    SELECT
        merchant_city,
        CASE NTILE(3) OVER (ORDER BY city_total_amount)
            WHEN 1 THEN 'Low Level'
            WHEN 2 THEN 'Middle Level'
            WHEN 3 THEN 'High Level'
        END AS city_amount_level
    FROM city_totals
)

SELECT
    t.*,
    COALESCE(c.city_amount_level, 'Città non disponibile') AS city_amount_level
FROM payments.default.pay_transactions AS t
LEFT JOIN city_groups AS c
    ON t.merchant_city = c.merchant_city;

**Check:** cities and transactions in each level.

In [0]:
%sql
SELECT city_amount_level,
       COUNT(DISTINCT merchant_city)  AS cities,
       COUNT(*)                       AS transactions,
       ROUND(SUM(amount), 0)          AS total_amount
FROM payments.default.pay_transactions
GROUP BY city_amount_level
ORDER BY total_amount DESC

### 2.3 Customer segmentation (Boston-matrix style)

**What:** gives each customer three labels, based on **how often** they buy (frequency) and **how much** they spend (monetary value).

**How**, in four steps:
1. **`customer_aggregation`:** for each `client_id`, counts the transactions (`customer_n_transactions`) and adds up the amounts (`customer_total_amount`).
2. **`customer_scores`:** splits the customers into thirds twice with `NTILE(3)`: once by number of transactions (`frequency_tile`) and once by total amount (`monetary_tile`). In both cases 1 = bottom third and 3 = top third.
3. **`customer_segments`:** turns the tiles into readable labels:
   - `customer_frequency_level`: Bassa / Media / Alta frequenza
   - `customer_monetary_level`: Basso / Medio / Alto valore
   - `customer_boston_class`: combines the two, as in the matrix below
4. **Final `SELECT`:** adds the three labels to every transaction of the customer. Customers that could not be scored get `'Non classificato'`.

**The segments** (`customer_boston_class`):

| | Value: Basso (1) | Value: Medio (2) | Value: Alto (3) |
|---|---|---|---|
| **Frequency: Alta (3)** | Frequent Value Seekers | Frequent Value Seekers | **Core Champions** |
| **Frequency: Media (2)** | Stable | Stable | High-Value Occasional |
| **Frequency: Bassa (1)** | **Watchlist** | Stable | High-Value Occasional |

- **Core Champions:** buy often and spend a lot. These are the best customers.
- **Frequent Value Seekers:** buy often but spend little or a medium amount.
- **High-Value Occasional:** buy rarely but spend a lot.
- **Watchlist:** buy rarely and spend little. These customers are at risk of becoming inactive. The name does **not** mean they are suspected of fraud.
- **Stable:** all the other combinations.

**Why:** segments summarise thousands of transactions into a few customer types, which can then be compared (for example, fraud rate by segment).

**Things to know:** the segment is a customer attribute, so it is repeated on every transaction of that customer. It is computed on all the years loaded, so it changes if more years are added.

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS

WITH customer_aggregation AS (
    SELECT
        client_id,
        COUNT(*) AS customer_n_transactions,
        SUM(amount) AS customer_total_amount
    FROM payments.default.pay_transactions
    WHERE client_id IS NOT NULL
      AND amount IS NOT NULL
    GROUP BY client_id
),

customer_scores AS (
    SELECT
        client_id,
        customer_n_transactions,
        customer_total_amount,

        NTILE(3) OVER (
            ORDER BY customer_n_transactions
        ) AS frequency_tile,

        NTILE(3) OVER (
            ORDER BY customer_total_amount
        ) AS monetary_tile

    FROM customer_aggregation
),

customer_segments AS (
    SELECT
        client_id,

        CASE
            WHEN frequency_tile = 1 THEN 'Bassa frequenza'
            WHEN frequency_tile = 2 THEN 'Media frequenza'
            WHEN frequency_tile = 3 THEN 'Alta frequenza'
        END AS customer_frequency_level,

        CASE
            WHEN monetary_tile = 1 THEN 'Basso valore'
            WHEN monetary_tile = 2 THEN 'Medio valore'
            WHEN monetary_tile = 3 THEN 'Alto valore'
        END AS customer_monetary_level,

        CASE
            WHEN frequency_tile = 3
             AND monetary_tile = 3
                THEN 'Core Champions'

            WHEN frequency_tile = 3
             AND monetary_tile IN (1, 2)
                THEN 'Frequent Value Seekers'

            WHEN frequency_tile IN (1, 2)
             AND monetary_tile = 3
                THEN 'High-Value Occasional'

            WHEN frequency_tile = 1
             AND monetary_tile = 1
                THEN 'Watchlist'

            ELSE 'Stable'
        END AS customer_boston_class

    FROM customer_scores
)

SELECT
    t.*,

    COALESCE(
        s.customer_frequency_level,
        'Non classificato'
    ) AS customer_frequency_level,

    COALESCE(
        s.customer_monetary_level,
        'Non classificato'
    ) AS customer_monetary_level,

    COALESCE(
        s.customer_boston_class,
        'Non classificato'
    ) AS customer_boston_class

FROM payments.default.pay_transactions AS t

LEFT JOIN customer_segments AS s
    ON t.client_id = s.client_id;

**Check:** customers in each segment, with their average transactions and spending.

In [0]:
%sql
SELECT customer_boston_class,
       COUNT(DISTINCT client_id)                                     AS customers,
       ROUND(COUNT(*) / COUNT(DISTINCT client_id), 0)                AS avg_transactions_per_customer,
       ROUND(SUM(amount) / COUNT(DISTINCT client_id), 0)             AS avg_amount_per_customer
FROM payments.default.pay_transactions
GROUP BY customer_boston_class
ORDER BY avg_amount_per_customer DESC

**Check:** the full frequency × value matrix (number of customers in each cell).

In [0]:
%sql
SELECT customer_frequency_level,
       COUNT(DISTINCT CASE WHEN customer_monetary_level = 'Basso valore' THEN client_id END) AS basso_valore,
       COUNT(DISTINCT CASE WHEN customer_monetary_level = 'Medio valore' THEN client_id END) AS medio_valore,
       COUNT(DISTINCT CASE WHEN customer_monetary_level = 'Alto valore'  THEN client_id END) AS alto_valore
FROM payments.default.pay_transactions
GROUP BY customer_frequency_level
ORDER BY customer_frequency_level

---
## 3. User Features — `pay_users`

### 3.1 Cleaning the money columns

**What:** turns `yearly_income`, `total_debt` and `per_capita_income` from text into numbers.

**How:** the raw values look like `$59,696`. `REGEXP_REPLACE(..., '[$,]', '')` removes every `$` and `,`, leaving `59696`, and `TRY_CAST(... AS DOUBLE)` converts it into a number. `TRY_CAST` returns **NULL** instead of failing if a value cannot be converted.

**Why:** money stored as text cannot be added, divided or compared, so none of the ratios below would be possible.

The query lists the columns to keep explicitly: the three money columns are replaced by their cleaned version, and all the other columns are kept as they are.

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_users AS
SELECT
  id, address, gender, birth_month, birth_year, current_age, retirement_age,
  latitude, longitude, credit_score, num_credit_cards,
  TRY_CAST(REGEXP_REPLACE(yearly_income,     '[$,]', '') AS DOUBLE) AS yearly_income,
  TRY_CAST(REGEXP_REPLACE(total_debt,        '[$,]', '') AS DOUBLE) AS total_debt,
  TRY_CAST(REGEXP_REPLACE(per_capita_income, '[$,]', '') AS DOUBLE) AS per_capita_income
FROM payments.default.pay_users;

**Check:** the converted columns are numeric, with no values lost in the conversion.

In [0]:
%sql
SELECT COUNT(*)                                        AS users,
       SUM(CASE WHEN yearly_income IS NULL THEN 1 ELSE 0 END)     AS null_yearly_income,
       SUM(CASE WHEN total_debt IS NULL THEN 1 ELSE 0 END)        AS null_total_debt,
       SUM(CASE WHEN per_capita_income IS NULL THEN 1 ELSE 0 END) AS null_per_capita_income,
       ROUND(AVG(yearly_income), 0)                    AS avg_yearly_income,
       ROUND(AVG(total_debt), 0)                       AS avg_total_debt
FROM payments.default.pay_users

### 3.2 Financial ratios, bands and credit risk

This step adds seven columns in one query.

**Ratios.** `NULLIF(x, 0)` turns a zero into NULL, so a division by zero gives NULL instead of an error.

| Column | Formula | Meaning |
|--------|---------|---------|
| `debt_to_income` | total_debt / yearly_income | How many years of income the debt is worth. Above 1 = debt larger than a year's income |
| `income_vs_area` | yearly_income / per_capita_income | Income compared with the area average. 1 = average, 2 = double the average |
| `years_to_retirement` | retirement_age − current_age | Years left before retirement. Negative = already retired |
| `debt_per_card` | total_debt / num_credit_cards | Average debt for each card held |

**Bands:**

| `credit_band` (credit score) | | `generation` (birth year) | |
|---|---|---|---|
| below 580 | Poor | 1997 or later | Gen Z |
| 580–669 | Fair | 1981–1996 | Millennial |
| 670–739 | Good | 1965–1980 | Gen X |
| 740–799 | Very Good | before 1965 | Boomer |
| 800 or more | Excellent | | |

The credit bands follow the ranges of the FICO score used in the US.

**Credit risk** (`risk_segment`): conditions are checked in order, and the first one that is true decides the segment.

| Segment | Condition |
|---------|-----------|
| **Alto** | credit score below 620, **or** debt more than 1.5 × yearly income |
| **Medio** | credit score below 700, **or** debt more than 0.8 × yearly income |
| **Basso** | everyone else |

**Things to know:** a `CASE` without a matching condition goes to the `ELSE` branch. So a missing `credit_score` becomes *Excellent* and a missing `birth_year` becomes *Boomer*.

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_users AS
SELECT *,
  ROUND(total_debt / NULLIF(yearly_income,0), 2)            AS debt_to_income,
  ROUND(yearly_income / NULLIF(per_capita_income,0), 2)     AS income_vs_area,
  retirement_age - current_age                               AS years_to_retirement,
  ROUND(total_debt / NULLIF(num_credit_cards,0), 0)         AS debt_per_card,
  CASE WHEN credit_score < 580 THEN 'Poor'
       WHEN credit_score < 670 THEN 'Fair'
       WHEN credit_score < 740 THEN 'Good'
       WHEN credit_score < 800 THEN 'Very Good'
       ELSE 'Excellent' END                                  AS credit_band,
  CASE WHEN birth_year >= 1997 THEN 'Gen Z'
       WHEN birth_year >= 1981 THEN 'Millennial'
       WHEN birth_year >= 1965 THEN 'Gen X'
       ELSE 'Boomer' END                                     AS generation,
  CASE WHEN credit_score < 620 OR total_debt/NULLIF(yearly_income,0) > 1.5 THEN 'Alto'
       WHEN credit_score < 700 OR total_debt/NULLIF(yearly_income,0) > 0.8 THEN 'Medio'
       ELSE 'Basso' END                                      AS risk_segment
FROM payments.default.pay_users;

**Check:** users in each risk segment, with their average ratios.

In [0]:
%sql
SELECT risk_segment,
       COUNT(*)                            AS users,
       ROUND(AVG(credit_score), 0)         AS avg_credit_score,
       ROUND(AVG(debt_to_income), 2)       AS avg_debt_to_income,
       ROUND(AVG(income_vs_area), 2)       AS avg_income_vs_area,
       ROUND(AVG(debt_per_card), 0)        AS avg_debt_per_card
FROM payments.default.pay_users
GROUP BY risk_segment
ORDER BY avg_credit_score

**Check:** users by credit band and by generation.

In [0]:
%sql
SELECT 'credit_band' AS band_type, credit_band AS band, COUNT(*) AS users FROM payments.default.pay_users GROUP BY credit_band
UNION ALL
SELECT 'generation', generation, COUNT(*) FROM payments.default.pay_users GROUP BY generation
ORDER BY band_type, users DESC

### 3.3 Rule-based fraud risk: `fraud_risk_level`

**What:** labels each user as **Critical**, **Suspicious** or **Normal**, based on financial signals that can indicate fraud (for example a synthetic identity or a stolen identity).

**How:** conditions are checked in order.

| Level | Condition | Why it is a signal |
|-------|-----------|--------------------|
| **Critical** | debt more than 3 × yearly income | Debt far beyond what the income could repay |
| **Critical** | 6 or more cards **and** income below the area average | Many cards with a low income: possible credit "bust-out" |
| **Suspicious** | debt more than 2 × yearly income | Very high indebtedness |
| **Suspicious** | income more than 5 × the area average | Income unusually high for the area: possibly declared falsely |
| **Normal** | everyone else | |

**Things to know:** these thresholds are **business rules chosen by hand**, not learned from data. Before relying on them, compare them with the real fraud labels in `pay_labels`. Users with missing values fall into *Normal*.

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_users AS
SELECT *,
  CASE WHEN total_debt > yearly_income * 3
         OR (num_credit_cards >= 6 AND yearly_income < per_capita_income) THEN 'Critical'
       WHEN total_debt > yearly_income * 2
         OR yearly_income > per_capita_income * 5 THEN 'Suspicious'
       ELSE 'Normal' END AS fraud_risk_level
FROM payments.default.pay_users;

**Check:** users per fraud risk level, compared with the credit risk segment.

In [0]:
%sql
SELECT fraud_risk_level,
       COUNT(*)                                                  AS users,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)        AS pct_users,
       SUM(CASE WHEN risk_segment = 'Alto'  THEN 1 ELSE 0 END)   AS risk_alto,
       SUM(CASE WHEN risk_segment = 'Medio' THEN 1 ELSE 0 END)   AS risk_medio,
       SUM(CASE WHEN risk_segment = 'Basso' THEN 1 ELSE 0 END)   AS risk_basso
FROM payments.default.pay_users
GROUP BY fraud_risk_level
ORDER BY users

---
## 4. Final Check

### Rows in each table

In [0]:
%sql
SELECT 'pay_transactions' AS table_name, COUNT(*) AS row_count FROM payments.default.pay_transactions
UNION ALL SELECT 'pay_users',  COUNT(*) FROM payments.default.pay_users
UNION ALL SELECT 'pay_cards',  COUNT(*) FROM payments.default.pay_cards
UNION ALL SELECT 'pay_labels', COUNT(*) FROM payments.default.pay_labels

### Sample of the transformed transactions

In [0]:
%sql
SELECT id, date, client_id, amount, merchant_city,
       is_online, city_amount_level,
       customer_frequency_level, customer_monetary_level, customer_boston_class
FROM payments.default.pay_transactions
LIMIT 10

### Sample of the transformed users

In [0]:
%sql
SELECT id, current_age, credit_score, yearly_income, total_debt,
       debt_to_income, income_vs_area, debt_per_card, years_to_retirement,
       credit_band, generation, risk_segment, fraud_risk_level
FROM payments.default.pay_users
LIMIT 10

### Do the new features relate to real fraud?

Joins the transactions with the fraud labels to compare the fraud rate across channels and customer segments.

In [0]:
%sql
SELECT t.is_online,
       t.customer_boston_class,
       COUNT(*)                                                     AS labelled_transactions,
       SUM(CASE WHEN l.is_fraud = 'Yes' THEN 1 ELSE 0 END)          AS frauds,
       ROUND(AVG(CASE WHEN l.is_fraud = 'Yes' THEN 1.0 ELSE 0 END) * 100, 3) AS fraud_rate_pct
FROM payments.default.pay_transactions AS t
JOIN payments.default.pay_labels AS l
  ON t.id = l.transaction_id
GROUP BY t.is_online, t.customer_boston_class
ORDER BY fraud_rate_pct DESC

---
## Conclusions

_Write 3–5 sentences in your own words after running the notebook, for example:_
- How many transactions were loaded for 2010 and 2011, and what share is online?
- How are customers spread across the segments, and which segment spends most?
- What share of users is in the *Alto* credit risk segment, and what share is *Critical* for fraud risk?
- Is the fraud rate higher online, or in some customer segments?